In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# 1. Load and Target Engineering
df = pd.read_csv('final_customer_purchase_behavior.csv')

# Define 'Likely to Purchase' as customers with Frequency >= Median and Recency <= Median
median_freq = df['Frequency'].median()
median_rec = df['Recency'].median()
df['Likely_to_Purchase'] = ((df['Frequency'] >= median_freq) & (df['Recency'] <= median_rec)).astype(int)

# 2. Preprocessing
le = LabelEncoder()
df['City_Encoded'] = le.fit_transform(df['City'])

# Select Features and Target
X = df[['Company_Profit', 'City_Encoded']]
y = df['Likely_to_Purchase']

# Scale features for stability
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. Model Training (Random Forest)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=100, max_depth=4, random_state=42)
model.fit(X_train, y_train)

# 4. Model Evaluation & Scoring
y_probs = model.predict_proba(X_scaled)[:, 1]
df['Purchase_Likelihood_Score'] = y_probs # Assign probability to each customer

print(f"Mean ROC-AUC (Cross-Val): {cross_val_score(model, X_scaled, y, cv=5).mean():.2f}")
df.to_csv('customer_purchase_predictions.csv', index=False)

Mean ROC-AUC (Cross-Val): 0.70
